# 01 - 什么是目标检测

**学习目标：**
- 理解图像分类、目标检测、实例分割的区别
- 理解检测任务的输出格式：bounding box + class + confidence
- 了解目标检测的实际应用场景

---

## 1. 三种视觉任务的对比

| 任务 | 输入 | 输出 | 示例 |
|------|------|------|------|
| **图像分类** | 一张图 | 一个类别标签 | "这是一只猫" |
| **目标检测** | 一张图 | 多个框 + 类别 | "图中有一只猫和一只狗，分别在这里" |
| **实例分割** | 一张图 | 像素级掩码 | "这是猫的轮廓，这是狗的轮廓" |

目标检测是分类和分割之间的桥梁——它不仅告诉你图里**有什么**，还告诉你**在哪里**。

## 2. 目标检测的输出格式

一个检测结果包含三个要素：

```
┌─────────────────────────────────┐
│         bounding box            │  ← 位置：(x1, y1, x2, y2)
│   ┌─────────────────────┐      │
│   │ 🐱                  │      │
│   │    cat  0.95        │      │  ← 类别 + 置信度
│   └─────────────────────┘      │
└─────────────────────────────────┘
```

- **Bounding Box**: 目标的矩形边界，通常用左上角和右下角坐标表示
- **Class**: 目标的类别（如 cat, dog, car）
- **Confidence/Score**: 模型对这个检测结果的确信程度（0-1）

## 3. 动手实验：用预训练模型体验目标检测

先不管原理，我们直接用 ultralytics 的预训练模型来体验一下目标检测是什么效果。

In [ ]:
# 导入必要的库
from ultralytics import YOLO
from PIL import Image
import requests
from io import BytesIO

# 加载预训练的 YOLO11 nano 模型
# 'n' = nano，最小的模型，适合学习和快速实验
# 首次运行会自动下载模型权重（约 5MB）
model = YOLO("yolo11n.pt")
print(f"模型加载完成！类别数: {len(model.names)}")
print(f"类别列表: {list(model.names.values())[:10]}...")

In [ ]:
# 下载一张示例图片
url = "https://ultralytics.com/images/bus.jpg"
response = requests.get(url)
image = Image.open(BytesIO(response.content))
print(f"图片尺寸: {image.size}")
image

In [ ]:
# 进行目标检测
results = model(image)

# results 是一个列表（因为可以一次处理多张图）
# 我们取第一张图的结果
result = results[0]

print(f"检测到 {len(result.boxes)} 个目标:\n")
for box in result.boxes:
    class_id = int(box.cls[0])
    class_name = model.names[class_id]
    confidence = float(box.conf[0])
    coords = box.xyxy[0].tolist()  # [x1, y1, x2, y2]
    print(f"  {class_name}: {confidence:.2%} 置信度")
    print(f"    位置: [{coords[0]:.0f}, {coords[1]:.0f}, {coords[2]:.0f}, {coords[3]:.0f}]")

In [ ]:
# 可视化检测结果
# ultralytics 内置了绘图方法
annotated = result.plot()  # 返回 numpy 数组
Image.fromarray(annotated[..., ::-1])  # BGR → RGB

## 4. 理解输出结构

ultralytics 的 `results` 对象封装了所有信息。让我们解剖一下它的结构：

In [ ]:
import numpy as np

print("=" * 50)
print("Result 对象结构")
print("=" * 50)

# 检测框坐标 (xyxy 格式: 左上角和右下角)
print(f"\nboxes.xyxy shape: {result.boxes.xyxy.shape}")
print(f"  → {len(result.boxes)} 个检测框，每个框 4 个坐标")
print(f"  示例: {result.boxes.xyxy[0].tolist()}")

# 置信度
print(f"\nboxes.conf shape: {result.boxes.conf.shape}")
print(f"  → 每个框一个置信度分数")
print(f"  示例: {result.boxes.conf[0].item():.4f}")

# 类别 ID
print(f"\nboxes.cls shape: {result.boxes.cls.shape}")
print(f"  → 每个框一个类别 ID")
print(f"  示例: {result.boxes.cls[0].item()} → {model.names[int(result.boxes.cls[0])]}")

# 类别名映射
print(f"\n模型类别数: {len(model.names)}")
print(f"前 5 个类别: {dict(list(model.names.items())[:5])}")

## 5. YOLO 的名字从何而来

**YOLO = You Only Look Once**

为什么叫"只看一次"？因为 YOLO 是**单阶段检测器**：

```
两阶段检测器 (如 Faster R-CNN):
  图片 → [阶段1] 生成候选区域 → [阶段2] 分类每个区域 → 结果
         (Region Proposal)        (Classification)

单阶段检测器 (YOLO):
  图片 → [一次前向传播] → 直接输出所有检测结果
         (One Pass)
```

**优势**: 速度快！YOLO 可以实时处理视频（>30 FPS）。

**代价**: 早期版本在小目标检测上不如两阶段方法（但 YOLOv8 已经大幅改进）。

## 6. 目标检测的应用场景

| 领域 | 应用 | 检测什么 |
|------|------|----------|
| 自动驾驶 | 行车环境感知 | 车辆、行人、交通标志 |
| 安防监控 | 异常行为检测 | 人、物品、行为 |
| 医疗影像 | 辅助诊断 | 病灶、器官、细胞 |
| 工业质检 | 缺陷检测 | 划痕、裂纹、异物 |
| 零售 | 智能结算 | 商品识别 |
| 农业 | 精准农业 | 病虫害、成熟度 |

## 7. 练习

### 练习 1: 用自己的图片测试
把一张你自己的图片放到 `data/` 目录，用上面的代码检测一下。

### 练习 2: 调整置信度阈值
在 `model(image, conf=0.5)` 中调整 `conf` 参数，观察检测结果的变化。

### 练习 3: 探索不同模型大小
试试 `yolo11s.pt` (small) 和 `yolo11m.pt` (medium)，对比检测效果和速度。

In [ ]:
# 练习 2: 不同置信度阈值的影响
print("置信度阈值对检测结果的影响:\n")
for conf_threshold in [0.25, 0.5, 0.75, 0.9]:
    results = model(image, conf=conf_threshold, verbose=False)
    n_detections = len(results[0].boxes)
    print(f"  conf={conf_threshold}: 检测到 {n_detections} 个目标")

---

**下一节**: [02 - 数据集与标注](02_dataset_and_annotations.ipynb) - 了解 YOLO 如何组织训练数据

**参考文档**: [docs/yolo_theory.md](../docs/yolo_theory.md) - YOLO 核心理论